## Hyperparameter Tuning with Full Metric Logging

Optuna backbone search on the identity variant, then per variant encoding search. Logs rank correlation, quintile Sharpe, score weighted Sharpe, hit rate, and training loss at every epoch of every trial. All metrics and epoch histories saved to disk.

### 1. Setup

In [ ]:
import gc
import json
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}, device: {device}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

### 2. Configuration

In [ ]:
country = 'IND'
raw_path = Path('../data/Global Factor_IND.parquet')
results_dir = Path('../results') / country / 'tuning'
results_dir.mkdir(parents = True, exist_ok = True)

coverage_threshold = 0.70
max_miss_frac = 1.0 / 3.0
min_stocks = 30
train_end = '2014-12-31'
val_end = '2019-12-31'

n_backbone_trials = 50
n_encoding_trials = 20
tuning_seed = 0
patience = 10
max_epochs = 50

load_always = ['id', 'gvkey', 'eom', 'excntry', 'ret_exc_lead1m', 'me']
exclude_cols = {
	'id', 'gvkey', 'iid', 'permno', 'permco', 'date', 'eom', 'excntry',
	'size_grp', 'obs_main', 'exch_main', 'common', 'primary_sec',
	'source_crsp', 'comp_tpci', 'crsp_shrcd', 'comp_exchg', 'crsp_exchcd',
	'curcd', 'fx', 'adjfct', 'bidask',
	'ret', 'ret_local', 'ret_exc', 'ret_exc_lead1m',
	'prc', 'prc_local', 'prc_high', 'prc_low',
	'me', 'me_company', 'dolvol', 'shares', 'tvol',
	'ret_lag_dif', 'div_tot',
}

print(f'country: {country}, backbone trials: {n_backbone_trials}')

### 3. Data Processing

In [ ]:
schema = pq.read_schema(raw_path)
all_col_names = schema.names
char_candidate = [c for c in all_col_names if c not in exclude_cols and c not in load_always]
needed = [c for c in load_always + char_candidate if c in all_col_names]
df = pd.read_parquet(raw_path, columns = needed)
df['eom'] = pd.to_datetime(df['eom'])
for col in char_candidate:
	if col in df.columns and df[col].dtype == np.float64: df[col] = df[col].astype(np.float32)
char_candidate = [c for c in char_candidate if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]

df_tr = df[df['eom'] <= train_end]
coverage = df_tr[char_candidate].notna().mean()
char_cols = sorted([c for c in char_candidate if coverage[c] >= coverage_threshold])
d = len(char_cols)
print(f'd = {d}')

id_cols = [c for c in load_always if c in df.columns]
df = df[id_cols + char_cols]
del df_tr
n_miss = df[char_cols].isna().sum(axis = 1)
df = df[n_miss <= d * max_miss_frac].reset_index(drop = True)

train_data = {}
val_data = {}
for eom in sorted(df['eom'].unique()):
	month = df[df['eom'] == eom].copy()
	if len(month) < min_stocks: continue
	ranks = month[char_cols].rank(pct = True, axis = 0) - 0.5
	month[char_cols] = ranks.fillna(0.0)
	x = month[char_cols].values.astype(np.float32)
	r = month['ret_exc_lead1m'].values.astype(np.float32)
	ids = month['id'].values
	hr = np.isfinite(r)
	if hr.sum() < 5: continue
	entry = {'x': x[hr], 'r': r[hr], 'ids': ids[hr]}
	if eom <= pd.Timestamp(train_end): train_data[eom] = entry
	elif eom <= pd.Timestamp(val_end): val_data[eom] = entry

del df
gc.collect()

def to_gpu(md):
	return {eom: {'x': torch.tensor(m['x'], dtype = torch.float32, device = device),
		'r': torch.tensor(m['r'], dtype = torch.float32, device = device),
		'ids': m['ids']} for eom, m in md.items()}

train_gpu = to_gpu(train_data)
val_gpu = to_gpu(val_data)
print(f'train: {len(train_gpu)}, val: {len(val_gpu)}, ~{np.mean([m["x"].shape[0] for m in train_gpu.values()]):.0f} firms')

### 4. Model Components

In [ ]:
class IdentityEncoder(nn.Module):
	def forward(self, x): return x

class LinearEncoder(nn.Module):
	def __init__(self, n):
		super().__init__()
		self.w = nn.Parameter(torch.ones(n))
		self.b = nn.Parameter(torch.zeros(n))
	def forward(self, x): return x * self.w + self.b

class PLEEncoder(nn.Module):
	def __init__(self, n, bins = 16):
		super().__init__()
		bd = torch.linspace(-0.5, 0.5, bins + 1)
		self.register_buffer('lo', bd[:-1])
		self.register_buffer('hi', bd[1:])
		self.w = nn.Parameter(torch.zeros(n, bins))
	def forward(self, x):
		a = torch.clamp((x.unsqueeze(-1) - self.lo) / (self.hi - self.lo + 1e-8), 0, 1)
		return x + (a * self.w.unsqueeze(0)).sum(-1)

class PeriodicEncoder(nn.Module):
	def __init__(self, n, nf = 8):
		super().__init__()
		self.om = nn.Parameter(torch.randn(n, nf))
		self.ph = nn.Parameter(torch.randn(n, nf) * 0.1)
		self.c = nn.Parameter(torch.zeros(n, nf))
	def forward(self, x):
		return x + (torch.sin(x.unsqueeze(-1) * self.om.unsqueeze(0) + self.ph.unsqueeze(0)) * self.c.unsqueeze(0)).sum(-1)

class FourierEncoder(nn.Module):
	def __init__(self, n, nf = 8):
		super().__init__()
		self.register_buffer('freq', torch.arange(1, nf + 1, dtype = torch.float32) * torch.pi)
		self.a = nn.Parameter(torch.zeros(n, nf))
		self.b = nn.Parameter(torch.zeros(n, nf))
	def forward(self, x):
		s = x.unsqueeze(-1) * self.freq
		return x + (torch.sin(s) * self.a.unsqueeze(0) + torch.cos(s) * self.b.unsqueeze(0)).sum(-1)

class MagnitudeDirectionEncoder(nn.Module):
	def __init__(self, n):
		super().__init__()
		self.wp = nn.Parameter(torch.ones(n))
		self.wn = nn.Parameter(torch.ones(n))
		self.b = nn.Parameter(torch.zeros(n))
	def forward(self, x): return F.relu(x) * self.wp - F.relu(-x) * self.wn + self.b

class AttentionHead(nn.Module):
	def __init__(self, n, s):
		super().__init__()
		self.w = nn.Parameter(torch.randn(n, n) * s)
		self.v = nn.Parameter(torch.randn(n, n) * s)
		self.sc = 1.0 / np.sqrt(n)
	def forward(self, y):
		return F.softmax((y @ self.w @ y.t()) * self.sc, dim = -1) @ (y @ self.v)

class TransformerBlock(nn.Module):
	def __init__(self, n, h, ff, s):
		super().__init__()
		self.heads = nn.ModuleList([AttentionHead(n, s) for _ in range(h)])
		self.w1 = nn.Parameter(torch.randn(n, ff) * (1.0 / ff))
		self.b1 = nn.Parameter(torch.zeros(ff))
		self.w2 = nn.Parameter(torch.randn(ff, n) * s)
		self.b2 = nn.Parameter(torch.zeros(n))
	def forward(self, y):
		y = sum(h(y) for h in self.heads) + y
		return F.relu(y @ self.w1 + self.b1) @ self.w2 + self.b2 + y

class PortfolioTransformer(nn.Module):
	def __init__(self, n, nb, nh, ff, enc):
		super().__init__()
		self.enc = enc
		s = 1.0 / n
		self.blocks = nn.ModuleList([TransformerBlock(n, nh, ff, s) for _ in range(nb)])
		self.lam = nn.Parameter(torch.randn(n) * s)
	def forward(self, x):
		y = self.enc(x)
		for b in self.blocks: y = b(y)
		return y @ self.lam
	def msrr_loss(self, x, r): return (1.0 - self.forward(x) @ r) ** 2

def build_encoder(v, n, **kwargs):
	if v == 'identity': return IdentityEncoder()
	elif v == 'linear': return LinearEncoder(n)
	elif v == 'ple': return PLEEncoder(n, bins = kwargs.get('ple_bins', 16))
	elif v == 'periodic': return PeriodicEncoder(n, nf = kwargs.get('periodic_freq', 8))
	elif v == 'fourier': return FourierEncoder(n, nf = kwargs.get('fourier_freq', 8))
	elif v == 'magnitude_dir': return MagnitudeDirectionEncoder(n)

### 5. Metric Computation

In [ ]:
@torch.no_grad()
def compute_all_metrics(model, data_gpu):
	"""Compute rank correlation, hit rate, quintile spread, score weighted
	return, and quintile Sharpe on a set of monthly cross sections."""
	model.eval()
	rank_corrs = []
	hit_rates = []
	quintile_spreads = []
	score_wt_rets = []
	quintile_rets = []

	for m in data_gpu.values():
		w = model(m['x']).cpu().numpy()
		r = m['r'].cpu().numpy()
		if len(w) < 10: continue

		# Rank correlation
		c, _ = spearmanr(w, r)
		if not np.isnan(c): rank_corrs.append(c)

		# Hit rate: fraction where sign of w matches sign of r
		w_dm = w - w.mean()
		hit = float(np.mean(np.sign(w_dm) == np.sign(r - r.mean())))
		hit_rates.append(hit)

		# Quintile spread: mean return of top quintile minus bottom quintile
		nq = max(1, int(len(w) * 0.20))
		so = np.argsort(w)
		top_ret = float(r[so[::-1][:nq]].mean())
		bot_ret = float(r[so[:nq]].mean())
		quintile_spreads.append(top_ret - bot_ret)
		quintile_rets.append(top_ret - bot_ret)

		# Score weighted return
		abs_sum = np.abs(w_dm).sum()
		if abs_sum > 1e-10:
			score_wt_rets.append(float((w_dm / abs_sum) @ r))

	metrics = {
		'rank_corr': float(np.mean(rank_corrs)) if rank_corrs else 0.0,
		'hit_rate': float(np.mean(hit_rates)) if hit_rates else 0.5,
		'quintile_spread': float(np.mean(quintile_spreads)) if quintile_spreads else 0.0,
	}

	# Annualised Sharpe from monthly returns
	if len(quintile_rets) > 1:
		qr = np.array(quintile_rets)
		metrics['quintile_sharpe'] = float(qr.mean() / max(qr.std(), 1e-8) * np.sqrt(12))
	else:
		metrics['quintile_sharpe'] = 0.0

	if len(score_wt_rets) > 1:
		sr = np.array(score_wt_rets)
		metrics['scorewt_sharpe'] = float(sr.mean() / max(sr.std(), 1e-8) * np.sqrt(12))
	else:
		metrics['scorewt_sharpe'] = 0.0

	model.train()
	return metrics

### 6. Training with Full Logging

In [ ]:
def train_with_logging(encoder, n_blocks_val, d_ff_val, lr_val, wd_val, seed = 0):
	"""Train one model, log all metrics at every epoch.
	Returns (best_val_corr, epoch_history, best_metrics)."""
	torch.manual_seed(seed)
	np.random.seed(seed)
	model = PortfolioTransformer(d, n_blocks_val, 1, d_ff_val, encoder).to(device)
	opt = torch.optim.Adam(model.parameters(), lr = lr_val, weight_decay = wd_val)
	keys = list(train_gpu.keys())
	nm = len(keys)

	epoch_history = []
	best_val_corr = -np.inf
	best_metrics = {}
	best_epoch = 0
	wait = 0

	for ep in range(1, max_epochs + 1):
		model.train()
		epoch_loss = 0.0
		for idx in np.random.permutation(nm):
			opt.zero_grad()
			loss = model.msrr_loss(train_gpu[keys[idx]]['x'], train_gpu[keys[idx]]['r'])
			loss.backward()
			nn.utils.clip_grad_norm_(model.parameters(), 1.0)
			opt.step()
			epoch_loss += loss.item()

		# Compute all metrics on train and val
		train_metrics = compute_all_metrics(model, train_gpu)
		val_metrics = compute_all_metrics(model, val_gpu)

		epoch_entry = {
			'epoch': ep,
			'train_loss': epoch_loss / nm,
			'train_rank_corr': train_metrics['rank_corr'],
			'train_hit_rate': train_metrics['hit_rate'],
			'train_quintile_spread': train_metrics['quintile_spread'],
			'train_quintile_sharpe': train_metrics['quintile_sharpe'],
			'train_scorewt_sharpe': train_metrics['scorewt_sharpe'],
			'val_rank_corr': val_metrics['rank_corr'],
			'val_hit_rate': val_metrics['hit_rate'],
			'val_quintile_spread': val_metrics['quintile_spread'],
			'val_quintile_sharpe': val_metrics['quintile_sharpe'],
			'val_scorewt_sharpe': val_metrics['scorewt_sharpe'],
		}
		epoch_history.append(epoch_entry)

		if val_metrics['rank_corr'] > best_val_corr:
			best_val_corr = val_metrics['rank_corr']
			best_metrics = val_metrics
			best_epoch = ep
			wait = 0
		else:
			wait += 1
		if wait >= patience: break

	best_metrics['best_epoch'] = best_epoch
	del model
	gc.collect()
	torch.cuda.empty_cache()
	return best_val_corr, epoch_history, best_metrics

### 7. Backbone Tuning (Identity Variant)

In [ ]:
all_trial_data = []

def backbone_objective(trial):
	lr_val = trial.suggest_float('lr', 1e-6, 1e-3, log = True)
	wd_val = trial.suggest_float('weight_decay', 1e-5, 1e-1, log = True)
	nb_val = trial.suggest_int('n_blocks', 1, 3)
	ff_val = trial.suggest_categorical('d_ff', [128, 256, 512])

	enc = IdentityEncoder().to(device)
	bvc, hist, metrics = train_with_logging(enc, nb_val, ff_val, lr_val, wd_val, seed = tuning_seed)

	# Store full trial data
	all_trial_data.append({
		'trial': trial.number,
		'params': {'lr': lr_val, 'weight_decay': wd_val, 'n_blocks': nb_val, 'd_ff': ff_val},
		'best_val_corr': bvc,
		'best_metrics': metrics,
		'epoch_history': hist,
	})

	# Log additional metrics as Optuna user attributes
	trial.set_user_attr('val_quintile_sharpe', metrics.get('quintile_sharpe', 0))
	trial.set_user_attr('val_scorewt_sharpe', metrics.get('scorewt_sharpe', 0))
	trial.set_user_attr('val_hit_rate', metrics.get('hit_rate', 0))
	trial.set_user_attr('best_epoch', metrics.get('best_epoch', 0))
	trial.set_user_attr('n_epochs_run', len(hist))

	if trial.number % 5 == 0:
		print(f'  trial {trial.number}: corr = {bvc:.4f}, Q = {metrics.get("quintile_sharpe", 0):.3f}, '
			f'SW = {metrics.get("scorewt_sharpe", 0):.3f}, ep = {metrics.get("best_epoch", 0)}')

	return bvc


print(f'Backbone search ({n_backbone_trials} trials)...')
backbone_study = optuna.create_study(direction = 'maximize', study_name = f'{country}_backbone')
backbone_study.optimize(backbone_objective, n_trials = n_backbone_trials)

bp = backbone_study.best_params
print(f'\nBest backbone (val corr = {backbone_study.best_value:.4f}):')
for k, v in bp.items(): print(f'  {k} = {v}')

### 8. Encoding Tuning

In [ ]:
best_lr = bp['lr']
best_wd = bp['weight_decay']
best_nb = bp['n_blocks']
best_ff = bp['d_ff']

encoding_results = {}
encoding_histories = {}

# Run each variant with best backbone
variants_to_tune = {
	'identity': {},
	'linear': {},
	'magnitude_dir': {},
}

for vname, kwargs in variants_to_tune.items():
	print(f'\n{vname}...')
	enc = build_encoder(vname, d, **kwargs).to(device)
	bvc, hist, metrics = train_with_logging(enc, best_nb, best_ff, best_lr, best_wd, seed = tuning_seed)
	encoding_results[vname] = {'val_corr': bvc, 'metrics': metrics, 'params': kwargs}
	encoding_histories[vname] = hist
	print(f'  corr: {bvc:.4f}, Q: {metrics["quintile_sharpe"]:.3f}, SW: {metrics["scorewt_sharpe"]:.3f}')


# PLE: tune bins
print('\nple tuning...')
ple_trials = []
def ple_objective(trial):
	bins = trial.suggest_categorical('ple_bins', [8, 16, 32, 64])
	enc = PLEEncoder(d, bins = bins).to(device)
	bvc, hist, metrics = train_with_logging(enc, best_nb, best_ff, best_lr, best_wd, seed = tuning_seed)
	ple_trials.append({'trial': trial.number, 'bins': bins, 'val_corr': bvc, 'metrics': metrics, 'history': hist})
	trial.set_user_attr('quintile_sharpe', metrics.get('quintile_sharpe', 0))
	trial.set_user_attr('scorewt_sharpe', metrics.get('scorewt_sharpe', 0))
	return bvc

ple_study = optuna.create_study(direction = 'maximize')
ple_study.optimize(ple_objective, n_trials = n_encoding_trials)
best_ple = ple_study.best_params
best_ple_trial = max(ple_trials, key = lambda x: x['val_corr'])
encoding_results['ple'] = {'val_corr': ple_study.best_value, 'metrics': best_ple_trial['metrics'], 'params': best_ple}
encoding_histories['ple'] = best_ple_trial['history']
print(f'  best bins = {best_ple["ple_bins"]}, corr = {ple_study.best_value:.4f}')


# Periodic: tune freq
print('\nperiodic tuning...')
per_trials = []
def periodic_objective(trial):
	nf = trial.suggest_categorical('periodic_freq', [4, 8, 16, 32])
	enc = PeriodicEncoder(d, nf = nf).to(device)
	bvc, hist, metrics = train_with_logging(enc, best_nb, best_ff, best_lr, best_wd, seed = tuning_seed)
	per_trials.append({'trial': trial.number, 'freq': nf, 'val_corr': bvc, 'metrics': metrics, 'history': hist})
	trial.set_user_attr('quintile_sharpe', metrics.get('quintile_sharpe', 0))
	return bvc

per_study = optuna.create_study(direction = 'maximize')
per_study.optimize(periodic_objective, n_trials = n_encoding_trials)
best_per = per_study.best_params
best_per_trial = max(per_trials, key = lambda x: x['val_corr'])
encoding_results['periodic'] = {'val_corr': per_study.best_value, 'metrics': best_per_trial['metrics'], 'params': best_per}
encoding_histories['periodic'] = best_per_trial['history']
print(f'  best freq = {best_per["periodic_freq"]}, corr = {per_study.best_value:.4f}')


# Fourier: tune freq
print('\nfourier tuning...')
fou_trials = []
def fourier_objective(trial):
	nf = trial.suggest_categorical('fourier_freq', [4, 8, 16, 32])
	enc = FourierEncoder(d, nf = nf).to(device)
	bvc, hist, metrics = train_with_logging(enc, best_nb, best_ff, best_lr, best_wd, seed = tuning_seed)
	fou_trials.append({'trial': trial.number, 'freq': nf, 'val_corr': bvc, 'metrics': metrics, 'history': hist})
	trial.set_user_attr('quintile_sharpe', metrics.get('quintile_sharpe', 0))
	return bvc

fou_study = optuna.create_study(direction = 'maximize')
fou_study.optimize(fourier_objective, n_trials = n_encoding_trials)
best_fou = fou_study.best_params
best_fou_trial = max(fou_trials, key = lambda x: x['val_corr'])
encoding_results['fourier'] = {'val_corr': fou_study.best_value, 'metrics': best_fou_trial['metrics'], 'params': best_fou}
encoding_histories['fourier'] = best_fou_trial['history']
print(f'  best freq = {best_fou["fourier_freq"]}, corr = {fou_study.best_value:.4f}')

### 9. Save Everything

In [ ]:
# Optuna backbone trial DataFrame
backbone_df = backbone_study.trials_dataframe()
backbone_df.to_csv(results_dir / f'{country}_backbone_trials.csv', index = False)

# Full trial data with epoch histories
with open(results_dir / f'{country}_backbone_trial_details.json', 'w') as f:
	json.dump(all_trial_data, f, default = float)

# Encoding results and epoch histories
with open(results_dir / f'{country}_encoding_results.json', 'w') as f:
	json.dump(encoding_results, f, indent = 2, default = float)

with open(results_dir / f'{country}_encoding_histories.json', 'w') as f:
	json.dump(encoding_histories, f, default = float)

# Consolidated summary
summary = {
	'country': country,
	'backbone': {'params': bp, 'val_corr': backbone_study.best_value},
	'encoding_comparison': {
		v: {
			'val_corr': r['val_corr'],
			'quintile_sharpe': r['metrics'].get('quintile_sharpe', 0),
			'scorewt_sharpe': r['metrics'].get('scorewt_sharpe', 0),
			'hit_rate': r['metrics'].get('hit_rate', 0),
			'best_epoch': r['metrics'].get('best_epoch', 0),
			'encoding_params': r.get('params', {}),
		}
		for v, r in encoding_results.items()
	},
}
with open(results_dir / f'{country}_tuning_summary.json', 'w') as f:
	json.dump(summary, f, indent = 2, default = float)

print('Saved files:')
for f in sorted(results_dir.glob(f'{country}_*')):
	print(f'  {f.name} ({f.stat().st_size / 1024:.0f} KB)')

### 10. Results Summary

In [ ]:
print(f'Tuning Summary: {country}')
print(f'\nBackbone: lr = {bp["lr"]:.6f}, wd = {bp["weight_decay"]:.6f}, '
	f'blocks = {bp["n_blocks"]}, d_ff = {bp["d_ff"]}')
print(f'\n{"Variant":<18} {"Corr":>6} {"Q SR":>6} {"SW SR":>6} {"Hit":>6} {"Epoch":>5} {"Params":>20}')
print()
for v in ['identity', 'linear', 'ple', 'periodic', 'fourier', 'magnitude_dir']:
	r = encoding_results.get(v, {})
	m = r.get('metrics', {})
	p = r.get('params', {})
	ps = ', '.join(f'{k} = {vv}' for k, vv in p.items()) if p else ''
	print(f'{v:<18} {r.get("val_corr", 0):6.4f} '
		f'{m.get("quintile_sharpe", 0):6.3f} '
		f'{m.get("scorewt_sharpe", 0):6.3f} '
		f'{m.get("hit_rate", 0):6.3f} '
		f'{m.get("best_epoch", 0):5d} '
		f'{ps:>20}')

print(f'\nCopy these to training notebooks:')
print(f'lr = {bp["lr"]}')
print(f'weight_decay = {bp["weight_decay"]}')
print(f'n_blocks = {bp["n_blocks"]}')
print(f'd_ff = {bp["d_ff"]}')

In [ ]:
# Training curves for best trial of each variant
fig, axes = plt.subplots(2, 3, figsize = (16, 10))
fig.suptitle(f'{country}: Training Curves (Best Configuration)', fontsize = 14)

for idx, vname in enumerate(['identity', 'linear', 'ple', 'periodic', 'fourier', 'magnitude_dir']):
	ax = axes[idx // 3, idx % 3]
	hist = encoding_histories.get(vname, [])
	if not hist: continue

	eps = [h['epoch'] for h in hist]
	ax.plot(eps, [h['train_rank_corr'] for h in hist], label = 'Train Corr', alpha = 0.7)
	ax.plot(eps, [h['val_rank_corr'] for h in hist], label = 'Val Corr', linewidth = 2)
	ax2 = ax.twinx()
	ax2.plot(eps, [h['train_loss'] for h in hist], color = 'red', alpha = 0.3, label = 'Loss')
	ax2.set_ylabel('Loss', color = 'red', fontsize = 8)

	ax.set_title(vname.replace('_', ' ').title(), fontsize = 10)
	ax.set_xlabel('Epoch', fontsize = 8)
	ax.set_ylabel('Rank Corr', fontsize = 8)
	ax.legend(fontsize = 7, loc = 'lower right')
	ax.grid(alpha = 0.3)

plt.tight_layout()
plt.savefig(results_dir / f'{country}_training_curves.png', dpi = 150, bbox_inches = 'tight')
plt.show()